In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (ToxTeller)

This notebook processes and standardizes the **ToxTeller** peptide dataset into a clean, consistent format for downstream analysis and machine learning workflows. The source provides two FASTA files corresponding to a **training set** and an **independent set**, which are merged into a single curated dataset.

- **Toxic effect / endpoint:** toxic
- **Source:** ToxTeller
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Reads raw FASTA files** from the training and independent splits.
- **Concatenates both splits** into one unified DataFrame.
- **Derives binary toxicity labels** from FASTA record identifiers:
  - sequences whose `id` contains `neg` → `label = 0` (non-toxic / negative)
  - otherwise → `label = 1` (toxic / positive)
- **Performs duplicate sequence quality control**:
  - consolidates repeated sequences,
  - flags sequences with conflicting labels as erroneous.
- **Builds dataset metadata** using the centralized raw-data description spreadsheet.
- **Exports the curated dataset and metadata** to the standardized output directory.

In [2]:
name_source = "ToxTeller"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
df_independent = read_fasta_doc(f"{PATH_INPUT}/{name_source}/independent_dataset.fasta")

In [4]:
df_train = read_fasta_doc(f"{PATH_INPUT}/{name_source}/training_dataset.fasta")

- Concatenate dataset

In [5]:
df_toxteller = (
    pd.concat([df_independent, df_train], 
              ignore_index=True)
)

In [6]:
df_toxteller = (
    df_toxteller
    .assign(
        label=lambda d: d["id"]
            .str.contains("neg", case=False, na=False)
            .map({True: 0, False: 1})
    )
    [["sequence","label"]]
)
df_toxteller.shape

(4329, 2)

- Checking duplicates

In [7]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df_toxteller, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

- Working with metada

In [8]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [9]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df_toxteller)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2024,
 'last update date': datetime.datetime(2023, 12, 25, 0, 0),
 'download date': Timestamp('2024-08-01 00:00:00'),
 'file format': 'fasta',
 'peptide property': 'toxic',
 'dataset information': 'Positive, Negative',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'Sampling from Swiss-Prot;No information',
 'repository or server': 'https://github.com/comics-asiis/ToxicPeptidePrediction',
 'publication': 'https://pubs.acs.org/doi/10.1021/acsomega.4c04246',
 'number_of_raw_sequences': 4329,
 'number_of_sequences_retained': 4329,
 'number_of_positive_sequences': 2078,
 'number_of_negative_sequences': 2251,
 'number_of_erroneous_sequences': 0,
 'modified_sequences_included': False}

- Exporting data

In [10]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [11]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_toxic_dataset.csv", index=False)